In [2]:

# Import libraries for reading CSV files and managing file paths.
import pandas as pd
from pathlib import Path
from IPython.display import display

# Identify the project directory.
# This works when Jupyter starts from the project root
# or from the notebooks subfolder.
project_root = Path.cwd().resolve()

if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

# Locate the original Olist CSV datasets.
raw_path = project_root / "data" / "raw"

# Find all CSV files and arrange them alphabetically.
csv_files = sorted(raw_path.glob("*.csv"))

# Verify that all nine datasets are available.
print("Dataset location:", raw_path)
print("Total CSV files found:", len(csv_files))

assert len(csv_files) == 9, "Expected 9 Olist CSV files."

# Store one summary row for each dataset.
dataset_summary = []

for file in csv_files:

    # Read the CSV file into pandas.
    df = pd.read_csv(file, low_memory=False)

    # Record the dataset's size and column names.
    dataset_summary.append({
        "Dataset": file.name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Column Names": ", ".join(df.columns)
    })

    # Release each DataFrame before loading the next file.
    del df

# Convert the results into a structured summary table.
summary_df = pd.DataFrame(dataset_summary)

# Display the dataset inventory.
display(summary_df)


Dataset location: C:\Users\naini\OneDrive\Documents\Olist-Ecommerce-Analytics\data\raw
Total CSV files found: 9


,Dataset,Rows,Columns,Column Names
0,olist_customers_dataset.csv,99441,5,"customer_id, customer_unique_id, customer_zip_..."
1,olist_geolocation_dataset.csv,1000163,5,"geolocation_zip_code_prefix, geolocation_lat, ..."
2,olist_order_items_dataset.csv,112650,7,"order_id, order_item_id, product_id, seller_id..."
3,olist_order_payments_dataset.csv,103886,5,"order_id, payment_sequential, payment_type, pa..."
4,olist_order_reviews_dataset.csv,99224,7,"review_id, order_id, review_score, review_comm..."
5,olist_orders_dataset.csv,99441,8,"order_id, customer_id, order_status, order_pur..."
6,olist_products_dataset.csv,32951,9,"product_id, product_category_name, product_nam..."
7,olist_sellers_dataset.csv,3095,4,"seller_id, seller_zip_code_prefix, seller_city..."
8,product_category_name_translation.csv,71,2,"product_category_name, product_category_name_e..."


In [3]:

# Create a list to store data-quality results for each CSV file.
quality_results = []

# Examine every source dataset individually.
for file in csv_files:

    # Load the original CSV without modifying the source.
    df = pd.read_csv(file, low_memory=False)

    # Count missing cells across all columns.
    missing_values = int(df.isna().sum().sum())

    # Count completely identical rows.
    duplicate_rows = int(df.duplicated().sum())

    # Calculate the percentage of missing cells.
    total_cells = df.shape[0] * df.shape[1]
    missing_percentage = (
        round(missing_values / total_cells * 100, 2)
        if total_cells > 0 else 0
    )

    # Record the data-quality metrics.
    quality_results.append({
        "Dataset": file.name,
        "Total Rows": len(df),
        "Missing Values": missing_values,
        "Missing %": missing_percentage,
        "Duplicate Rows": duplicate_rows
    })

    # Release memory after profiling the file.
    del df

# Create and display a summary of data-quality findings.
quality_df = pd.DataFrame(quality_results)

display(quality_df)


,Dataset,Total Rows,Missing Values,Missing %,Duplicate Rows
0,olist_customers_dataset.csv,99441,0,0.00,0
1,olist_geolocation_dataset.csv,1000163,0,0.00,261831
2,olist_order_items_dataset.csv,112650,0,0.00,0
3,olist_order_payments_dataset.csv,103886,0,0.00,0
4,olist_order_reviews_dataset.csv,99224,145903,21.01,0
5,olist_orders_dataset.csv,99441,4908,0.62,0
6,olist_products_dataset.csv,32951,2448,0.83,0
7,olist_sellers_dataset.csv,3095,0,0.00,0
8,product_category_name_translation.csv,71,0,0.00,0


In [4]:

# Analyze missing values at the column level for datasets
# that showed missing data in our initial quality checks.

import pandas as pd
from IPython.display import display

# Identify the datasets requiring closer investigation.
datasets_to_check = [
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv"
]

# Analyze each dataset separately.
for dataset_name in datasets_to_check:

    # Load the original dataset without changing its values.
    df = pd.read_csv(raw_path / dataset_name, low_memory=False)

    # Calculate missing-value counts and percentages
    # for every column.
    missing_summary = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Missing Count": df.isna().sum().values,
        "Missing %": (
            df.isna().mean().values * 100
        ).round(2)
    })

    # Sort columns from most missing to least missing.
    missing_summary = missing_summary.sort_values(
        by="Missing Count",
        ascending=False
    )

    # Display only columns that contain missing values.
    missing_summary = missing_summary[
        missing_summary["Missing Count"] > 0
    ]

    print(f"\nDataset: {dataset_name}")
    print(f"Total Rows: {len(df):,}")

    display(missing_summary)

    # Release memory after profiling this dataset.
    del df



Dataset: olist_order_reviews_dataset.csv
Total Rows: 99,224


,Column,Data Type,Missing Count,Missing %
3,review_comment_title,str,87656,88.34
4,review_comment_message,str,58247,58.70



Dataset: olist_orders_dataset.csv
Total Rows: 99,441


,Column,Data Type,Missing Count,Missing %
6,order_delivered_customer_date,str,2965,2.98
5,order_delivered_carrier_date,str,1783,1.79
4,order_approved_at,str,160,0.16



Dataset: olist_products_dataset.csv
Total Rows: 32,951


,Column,Data Type,Missing Count,Missing %
1,product_category_name,str,610,1.85
3,product_description_lenght,float64,610,1.85
2,product_name_lenght,float64,610,1.85
4,product_photos_qty,float64,610,1.85
5,product_weight_g,float64,2,0.01
7,product_height_cm,float64,2,0.01
6,product_length_cm,float64,2,0.01
8,product_width_cm,float64,2,0.01


In [5]:

# Investigate whether missing order timestamps are related
# to the current status of each e-commerce order.

import pandas as pd
from IPython.display import display

# Load the original orders dataset without modifying it.
orders = pd.read_csv(
    raw_path / "olist_orders_dataset.csv",
    low_memory=False
)

# Group orders by status and count missing timestamps.
status_summary = (
    orders.groupby("order_status", dropna=False)
    .agg(
        Total_Orders=("order_id", "count"),

        Missing_Approval_Date=(
            "order_approved_at",
            lambda x: x.isna().sum()
        ),

        Missing_Carrier_Date=(
            "order_delivered_carrier_date",
            lambda x: x.isna().sum()
        ),

        Missing_Delivery_Date=(
            "order_delivered_customer_date",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

# Calculate each order status's share of all orders.
status_summary["Order_Percentage"] = (
    status_summary["Total_Orders"]
    / len(orders) * 100
).round(2)

# Show the most common order statuses first.
status_summary = status_summary.sort_values(
    by="Total_Orders",
    ascending=False
)

# Display the results for business interpretation.
display(status_summary)


,order_status,Total_Orders,Missing_Approval_Date,Missing_Carrier_Date,Missing_Delivery_Date,Order_Percentage
3,delivered,96478,14,2,8,97.02
6,shipped,1107,0,0,1107,1.11
1,canceled,625,141,550,619,0.63
7,unavailable,609,0,609,609,0.61
4,invoiced,314,0,314,314,0.32
5,processing,301,0,301,301,0.30
2,created,5,5,5,5,0.01
0,approved,2,0,2,2,0.00


In [6]:

# Identify orders marked as delivered but without an actual delivery date.
delivered_missing_date = orders[
    (orders["order_status"] == "delivered")
    & (orders["order_delivered_customer_date"].isna())
].copy()

# Identify canceled orders that still have a recorded delivery date.
canceled_with_delivery = orders[
    (orders["order_status"] == "canceled")
    & (orders["order_delivered_customer_date"].notna())
].copy()

# Display counts for both unusual business conditions.
print("Delivered orders missing delivery date:",
      len(delivered_missing_date))

print("Canceled orders with delivery date:",
      len(canceled_with_delivery))

# Inspect delivery timestamps for the inconsistent delivered orders.
print("\nDelivered orders missing delivery dates:")
display(delivered_missing_date[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
])

# Inspect canceled orders that have a delivery timestamp.
print("\nCanceled orders with recorded delivery dates:")
display(canceled_with_delivery[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ]
])


Delivered orders missing delivery date: 8
Canceled orders with delivery date: 6

Delivered orders missing delivery dates:


,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00



Canceled orders with recorded delivery dates:


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-03-21 22:03:51
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-16 14:36:59
58266,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-07 14:52:30,2016-10-14 15:07:11
59332,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-08 20:17:50,2016-10-19 18:47:43
92636,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-10-03 21:01:41,2016-11-08 10:58:34
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,canceled,2016-10-09 15:39:56,2016-11-09 14:53:50


In [7]:

# Validate candidate primary keys in the Olist datasets.
# Check whether the key columns contain nulls or duplicates.

# Define candidate keys for each dataset.
candidate_keys = {
    "olist_customers_dataset.csv": ["customer_id"],
    "olist_orders_dataset.csv": ["order_id"],
    "olist_order_items_dataset.csv": ["order_id", "order_item_id"],
    "olist_order_payments_dataset.csv": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset.csv": ["review_id", "order_id"],
    "olist_products_dataset.csv": ["product_id"],
    "olist_sellers_dataset.csv": ["seller_id"],
    "product_category_name_translation.csv": ["product_category_name"]
}

# Create a list to store validation results.
key_results = []

# Validate each candidate key.
for filename, key_columns in candidate_keys.items():

    # Load the relevant dataset.
    df = pd.read_csv(raw_path / filename, low_memory=False)

    # Count rows with at least one missing key component.
    missing_keys = int(
        df[key_columns].isna().any(axis=1).sum()
    )

    # Count records with duplicate key combinations.
    duplicate_keys = int(
        df.duplicated(subset=key_columns).sum()
    )

    # Record the findings.
    key_results.append({
        "Dataset": filename,
        "Candidate Key": " + ".join(key_columns),
        "Total Rows": len(df),
        "Missing Key Rows": missing_keys,
        "Duplicate Key Rows": duplicate_keys,
        "Unique Candidate Key": (
            missing_keys == 0 and duplicate_keys == 0
        )
    })

    # Release memory before processing the next file.
    del df

# Display the candidate-key validation results.
key_df = pd.DataFrame(key_results)

display(key_df)


,Dataset,Candidate Key,Total Rows,Missing Key Rows,Duplicate Key Rows,Unique Candidate Key
0,olist_customers_dataset.csv,customer_id,99441,0,0,True
1,olist_orders_dataset.csv,order_id,99441,0,0,True
2,olist_order_items_dataset.csv,order_id + order_item_id,112650,0,0,True
3,olist_order_payments_dataset.csv,order_id + payment_sequential,103886,0,0,True
4,olist_order_reviews_dataset.csv,review_id + order_id,99224,0,0,True
5,olist_products_dataset.csv,product_id,32951,0,0,True
6,olist_sellers_dataset.csv,seller_id,3095,0,0,True
7,product_category_name_translation.csv,product_category_name,71,0,0,True


In [8]:

# Validate foreign-key relationships across the Olist datasets.
# Identify records referencing missing parent records.

import pandas as pd
from IPython.display import display

# Define relationships using:
# Child table, foreign key, parent table, primary key.
relationships = [
    (
        "olist_orders_dataset.csv", "customer_id",
        "olist_customers_dataset.csv", "customer_id"
    ),
    (
        "olist_order_items_dataset.csv", "order_id",
        "olist_orders_dataset.csv", "order_id"
    ),
    (
        "olist_order_items_dataset.csv", "product_id",
        "olist_products_dataset.csv", "product_id"
    ),
    (
        "olist_order_items_dataset.csv", "seller_id",
        "olist_sellers_dataset.csv", "seller_id"
    ),
    (
        "olist_order_payments_dataset.csv", "order_id",
        "olist_orders_dataset.csv", "order_id"
    ),
    (
        "olist_order_reviews_dataset.csv", "order_id",
        "olist_orders_dataset.csv", "order_id"
    ),
    (
        "olist_products_dataset.csv", "product_category_name",
        "product_category_name_translation.csv",
        "product_category_name"
    )
]

# Store results for every relationship.
fk_results = []

for child_file, fk, parent_file, pk in relationships:

    # Read only the columns needed for this validation.
    child = pd.read_csv(
        raw_path / child_file,
        usecols=[fk],
        dtype={fk: "string"}
    )

    parent = pd.read_csv(
        raw_path / parent_file,
        usecols=[pk],
        dtype={pk: "string"}
    )

    # Identify missing foreign-key values.
    missing_fk = child[fk].isna()

    # Identify non-null foreign keys absent from the parent.
    orphan_mask = (
        child[fk].notna()
        & ~child[fk].isin(parent[pk])
    )

    # Record relationship integrity metrics.
    fk_results.append({
        "Child Table": child_file.replace(".csv", ""),
        "Foreign Key": fk,
        "Parent Table": parent_file.replace(".csv", ""),
        "Total Records": len(child),
        "Missing FK": int(missing_fk.sum()),
        "Orphan Records": int(orphan_mask.sum()),
        "Referential Integrity": (
            "PASS" if orphan_mask.sum() == 0 else "REVIEW"
        )
    })

# Display the foreign-key validation report.
fk_df = pd.DataFrame(fk_results)

display(fk_df)


,Child Table,Foreign Key,Parent Table,Total Records,Missing FK,Orphan Records,Referential Integrity
0,olist_orders_dataset,customer_id,olist_customers_dataset,99441,0,0,PASS
1,olist_order_items_dataset,order_id,olist_orders_dataset,112650,0,0,PASS
2,olist_order_items_dataset,product_id,olist_products_dataset,112650,0,0,PASS
3,olist_order_items_dataset,seller_id,olist_sellers_dataset,112650,0,0,PASS
4,olist_order_payments_dataset,order_id,olist_orders_dataset,103886,0,0,PASS
5,olist_order_reviews_dataset,order_id,olist_orders_dataset,99224,0,0,PASS
6,olist_products_dataset,product_category_name,product_category_name_translation,32951,610,13,REVIEW


In [9]:

# Investigate product categories that are missing
# or do not have an English translation.
# Load product and category translation datasets.
products = pd.read_csv(
    raw_path / "olist_products_dataset.csv"
)

translations = pd.read_csv(
    raw_path / "product_category_name_translation.csv"
)

# Identify products without an assigned category.
missing_categories = products[
    products["product_category_name"].isna()
]

# Identify category values that are not in the translation table.
valid_categories = set(
    translations["product_category_name"].dropna()
)

unmatched_products = products[
    products["product_category_name"].notna()
    & ~products["product_category_name"].isin(valid_categories)
]

# Count missing and unmatched category records.
print("Products with missing categories:",
      len(missing_categories))

print("Products with unmatched categories:",
      len(unmatched_products))

# Count the distinct unmatched category names.
print(
    "Distinct unmatched categories:",
    unmatched_products["product_category_name"].nunique()
)

# Display unmatched categories and their product counts.
unmatched_summary = (
    unmatched_products["product_category_name"]
    .value_counts()
    .rename_axis("Unmatched Category")
    .reset_index(name="Product Count")
)

display(unmatched_summary)


Products with missing categories: 610
Products with unmatched categories: 13
Distinct unmatched categories: 2


,Unmatched Category,Product Count
0,portateis_cozinha_e_preparadores_de_alimentos,10
1,pc_gamer,3


In [10]:


# Locate the project root.
project_root = Path.cwd().resolve()

if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

# Create a folder for manually maintained reference data.
reference_path = project_root / "data" / "reference"
reference_path.mkdir(parents=True, exist_ok=True)

# Define custom English translations for the two
# categories missing from the original translation file.
category_overrides = pd.DataFrame({
    "product_category_name": [
        "portateis_cozinha_e_preparadores_de_alimentos",
        "pc_gamer"
    ],
    "product_category_name_english": [
        "Portable Kitchen & Food Preparation Appliances",
        "Gaming PCs"
    ]
})

# Save the approved reference overrides separately.
output_file = (
    reference_path / "product_category_translation_overrides.csv"
)

category_overrides.to_csv(output_file, index=False)

# Display the mapping and its saved location.
display(category_overrides)

print("Mapping file created successfully!")
print("Saved at:", output_file)


,product_category_name,product_category_name_english
0,portateis_cozinha_e_preparadores_de_alimentos,Portable Kitchen & Food Preparation Appliances
1,pc_gamer,Gaming PCs


Mapping file created successfully!
Saved at: C:\Users\naini\OneDrive\Documents\Olist-Ecommerce-Analytics\data\reference\product_category_translation_overrides.csv
